# Experimento 03 — Explicabilidad por instancia con SHAP

**Objetivo.** Calcular SHAP values sobre las anomalías detectadas por el Isolation Forest del Experimento 02 para descomponer cada score en la contribución de sus features y caracterizar tipologías de anomalía.

**Población.**
- 108 231 anomalías robustas (presentes en las 5 semillas del análisis de estabilidad de exp_02).
- 50 000 ofertas normales muestreadas de forma estratificada por segmento UNSPSC, como referencia para los plots agregados.

**Estrategia.** Para cada oferta se calcula SHAP en el modelo (segmento o clase) que efectivamente le aplicó. El feature engineering y los hiperparámetros se replican del notebook `exp_02_refinamiento/IF_SICOP_v2.ipynb` con `random_state=42`, lo que garantiza que los modelos reentrenados sean idénticos a los que produjeron los scores originales.

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from pathlib import Path
import time
import warnings
warnings.filterwarnings('ignore')

try:
    import shap
    print(f'shap version: {shap.__version__}')
except ImportError:
    raise ImportError('Se requiere shap. Instalar con: pip install shap')

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

DATA_PATH = Path('..') / 'datos' / 'V_BASE_OFERTA_ITEM.parquet'
ROBUSTAS_PATH = Path('..') / 'exp_02_refinamiento' / 'resultados' / 'anomalias_exp02_robustas.csv'
EXP02_PARQUET = Path('..') / 'exp_02_refinamiento' / 'resultados' / 'anomalias_exp02.parquet'

FIG_DIR = Path('figuras')
RES_DIR = Path('resultados')
FIG_DIR.mkdir(exist_ok=True)
RES_DIR.mkdir(exist_ok=True)

SEMILLA_PRINCIPAL = 42
MIN_OBS_CLASE = 500
N_NORMALES_MUESTRA = 50_000
# Máximo de filas por segmento/clase para TreeExplainer (SHAP). Sin tope, una sola clase muy poblada puede tardar horas.
# Siempre se intentan incluir todas las anomalías robustas del grupo; el remanente se llena con muestra normal.
MAX_SHAP_ROWS_PER_GROUP = 4000

df = pd.read_parquet(DATA_PATH)
print(f'Registros: {len(df):,}  |  Columnas: {df.shape[1]}')

shap version: 0.51.0
Registros: 3,021,336  |  Columnas: 70


## 2. Feature engineering (replica determinista de exp_02)

Los pasos son idénticos a las celdas 5–8 del notebook `IF_SICOP_v2.ipynb`. Se reproduce aquí para que los modelos reentrenados con `random_state=42` produzcan exactamente los mismos scores que en exp_02.

In [2]:
df['LOG_PRECIO_OFERTADO'] = np.where(
    df['OFERTA_PRECIO_UNITARIO_CRC'] > 0,
    np.log10(df['OFERTA_PRECIO_UNITARIO_CRC']), np.nan)
df['LOG_PRECIO_ESTIMADO'] = np.where(
    df['CARTEL_PRECIO_UNITARIO_CRC'] > 0,
    np.log10(df['CARTEL_PRECIO_UNITARIO_CRC']), np.nan)
df['LOG_RATIO_VS_ESTIMADO'] = np.where(
    df['RATIO_OFERTADO_VS_ESTIMADO'] > 0,
    np.log10(df['RATIO_OFERTADO_VS_ESTIMADO']), np.nan)

for col in ['FECHA_PUBLICACION', 'FECHA_CIERRE_RECEPCION',
            'FECHA_APERTURA', 'FECHA_PRESENTA_OFERTA']:
    df[col] = pd.to_datetime(df[col], errors='coerce')

df['DIAS_PARA_CIERRE'] = (
    df['FECHA_CIERRE_RECEPCION'] - df['FECHA_PRESENTA_OFERTA']
).dt.total_seconds() / 86400
df['DIAS_VENTANA_PROCESO'] = (
    df['FECHA_CIERRE_RECEPCION'] - df['FECHA_PUBLICACION']
).dt.total_seconds() / 86400

df['RATIO_CANTIDAD_VS_SOLICITADA'] = np.where(
    df['CARTEL_CANTIDAD_SOLICITADA'].isna() | (df['CARTEL_CANTIDAD_SOLICITADA'] == 0),
    np.nan, df['CANTIDAD_OFERTADA'] / df['CARTEL_CANTIDAD_SOLICITADA'])

oferta_prod_8 = df['OFERTA_CODIGO_PRODUCTO'].astype(str).str[:8]
cartel_prod_8 = df['CARTEL_CODIGO_PRODUCTO'].astype(str).str[:8]
df['PRODUCTO_DIFIERE'] = (oferta_prod_8 != cartel_prod_8).astype(int)

df['CV_PRECIO_ITEM'] = np.where(
    df['PRECIO_PROM_ITEM_CRC'].isna() | (df['PRECIO_PROM_ITEM_CRC'] == 0),
    np.nan, df['PRECIO_STDDEV_ITEM_CRC'] / df['PRECIO_PROM_ITEM_CRC'])

df['REGIMEN_LGCP'] = (df['FECHA_PRESENTA_OFERTA'] >= '2022-12-01').astype(int)

tamano_map = {'Microemprendedor': 1, 'Pequeña': 2, 'Mediana': 3, 'Grande': 4}
df['TAMANO_PROVEEDOR_ORD'] = df['TAMANO_PROVEEDOR'].map(tamano_map)

print('Feature engineering completado.')

Feature engineering completado.


In [3]:
tp_cols = [c for c in df.columns if c.startswith('TP_')]
mp_cols = [c for c in df.columns if c.startswith('MP_')]

FEATURES_RELATIVAS = [
    'RATIO_OFERTADO_VS_ESTIMADO', 'RATIO_OFERTADO_VS_PROM_ITEM',
    'RATIO_OFERTADO_VS_MIN_ITEM', 'LOG_RATIO_VS_ESTIMADO',
    'RANK_PRECIO_ASC', 'N_OFERTAS_ITEM', 'N_OFERENTES_ITEM',
    'N_OFERENTES_ELEGIBLES_ITEM', 'SINGLE_BID_FLAG', 'CV_PRECIO_ITEM',
    'RATIO_CANTIDAD_VS_SOLICITADA', 'PRODUCTO_DIFIERE',
    'DIAS_PARA_CIERRE', 'DIAS_VENTANA_PROCESO',
    'OFERTA_IVA', 'OFERTA_DESCUENTO', 'REGIMEN_LGCP',
] + tp_cols + mp_cols + ['TAMANO_PROVEEDOR_ORD']

FEATURES_ABSOLUTAS = [
    'LOG_PRECIO_OFERTADO', 'LOG_PRECIO_ESTIMADO', 'CANTIDAD_OFERTADA',
    'N_OFERTAS_ITEM', 'N_OFERENTES_ITEM', 'N_OFERENTES_ELEGIBLES_ITEM',
    'SINGLE_BID_FLAG', 'RATIO_OFERTADO_VS_ESTIMADO', 'LOG_RATIO_VS_ESTIMADO',
    'CV_PRECIO_ITEM', 'RATIO_CANTIDAD_VS_SOLICITADA', 'PRODUCTO_DIFIERE',
    'DIAS_PARA_CIERRE', 'DIAS_VENTANA_PROCESO',
    'OFERTA_IVA', 'OFERTA_DESCUENTO', 'REGIMEN_LGCP',
] + tp_cols + mp_cols + ['TAMANO_PROVEEDOR_ORD']

all_features = list(set(FEATURES_RELATIVAS + FEATURES_ABSOLUTAS))
binary_features = ['SINGLE_BID_FLAG', 'PRODUCTO_DIFIERE', 'REGIMEN_LGCP'] + tp_cols + mp_cols

for f in binary_features:
    if f in df.columns:
        df[f] = df[f].fillna(0)

numeric_features = [f for f in all_features if f not in binary_features]
for f in numeric_features:
    if f in df.columns and df[f].isna().any():
        df[f] = df[f].fillna(df[f].median())

if 'RANK_PRECIO_ASC' in df.columns:
    df['RANK_PRECIO_ASC'] = df['RANK_PRECIO_ASC'].astype(float)

print(f'Features RELATIVAS (Nivel 1): {len(FEATURES_RELATIVAS)}')
print(f'Features ABSOLUTAS (Nivel 2): {len(FEATURES_ABSOLUTAS)}')
print(f'Nulos restantes: {df[all_features].isna().sum().sum()}')

Features RELATIVAS (Nivel 1): 18
Features ABSOLUTAS (Nivel 2): 18
Nulos restantes: 0


## 3. Carga de anomalías robustas y muestra normal

El conjunto de anomalías robustas viene del análisis de estabilidad de exp_02 (5 semillas). Se cruza con el dataframe principal usando `(NRO_SICOP, NRO_OFERTA, NRO_LINEA)` como llave.

In [4]:
robustas = pd.read_csv(ROBUSTAS_PATH)
print(f'Anomalías robustas en CSV: {len(robustas):,}')
print(f'Columnas: {list(robustas.columns)[:10]}...')

key_cols = ['NRO_SICOP', 'NRO_OFERTA', 'NRO_LINEA']
for k in key_cols:
    if k in robustas.columns:
        df[k] = df[k].astype(robustas[k].dtype, errors='ignore')

df_keyed = df.set_index(key_cols)
robustas_keyed = robustas.set_index(key_cols)
robustas_idx = df_keyed.index.isin(robustas_keyed.index)
df['IS_ROBUST_ANOMALY'] = robustas_idx.astype(int)
n_rob = df['IS_ROBUST_ANOMALY'].sum()
print(f'Cruzadas con dataset principal: {n_rob:,} ({n_rob/len(df)*100:.2f}%)')

Anomalías robustas en CSV: 108,231
Columnas: ['NRO_SICOP', 'NRO_PROCEDIMIENTO', 'NRO_OFERTA', 'NRO_LINEA', 'CEDULA_INSTITUCION', 'NOMBRE_INSTITUCION', 'TIPO_PROCEDIMIENTO', 'MODALIDAD_PROCEDIMIENTO', 'CEDULA_PROVEEDOR', 'NOMBRE_PROVEEDOR']...
Cruzadas con dataset principal: 108,231 (3.58%)


In [5]:
rng = np.random.default_rng(SEMILLA_PRINCIPAL)
normales_pool = df[df['IS_ROBUST_ANOMALY'] == 0]

n_por_segmento = (
    normales_pool['OFERTA_SEGMENTO']
    .value_counts(normalize=True)
    .mul(N_NORMALES_MUESTRA)
    .round()
    .astype(int)
    .clip(lower=1)
)

muestras = []
for seg, n in n_por_segmento.items():
    sub = normales_pool[normales_pool['OFERTA_SEGMENTO'] == seg]
    n_take = min(n, len(sub))
    if n_take > 0:
        muestras.append(sub.sample(n=n_take, random_state=SEMILLA_PRINCIPAL))

muestra_normal = pd.concat(muestras)
print(f'Muestra normal estratificada: {len(muestra_normal):,} ofertas')
print(f'Segmentos cubiertos: {muestra_normal["OFERTA_SEGMENTO"].nunique()}')

df['IS_SAMPLE'] = ((df['IS_ROBUST_ANOMALY'] == 1) | df.index.isin(muestra_normal.index)).astype(int)
print(f'\nPoblación SHAP total: {df["IS_SAMPLE"].sum():,}')
print(f'  Anomalías robustas: {(df["IS_SAMPLE"]==1).sum() - len(muestra_normal):,}')
print(f'  Normales muestreadas: {len(muestra_normal):,}')

Muestra normal estratificada: 49,999 ofertas
Segmentos cubiertos: 57

Población SHAP total: 158,230
  Anomalías robustas: 108,231
  Normales muestreadas: 49,999


## 4. Reentrenamiento determinista y SHAP por segmento (Nivel 1)

Para cada segmento que contiene al menos una anomalía robusta o muestra normal, se reentrena el `IsolationForest` con `random_state=42` (idéntico a exp_02) y se calcula SHAP **solo** sobre las observaciones de ese segmento que están en la población SHAP. Esto evita calcular SHAP sobre los 3M de filas y mantiene la equivalencia con el modelo original.

In [6]:
IF_PARAMS = dict(
    n_estimators=200, max_samples='auto',
    contamination='auto', random_state=SEMILLA_PRINCIPAL, n_jobs=-1,
)

shap_seg_records = []
segmentos = sorted(df['OFERTA_SEGMENTO'].unique())
n_segmentos_relevantes = sum(1 for s in segmentos if df.loc[df['OFERTA_SEGMENTO']==s, 'IS_SAMPLE'].sum() > 0)
print(f'Segmentos con observaciones en población SHAP: {n_segmentos_relevantes}')

t0 = time.time()
for i, seg in enumerate(segmentos):
    mask_full = (df['OFERTA_SEGMENTO'] == seg).values
    mask_sample = mask_full & (df['IS_SAMPLE'].values == 1)
    n_full = mask_full.sum()
    n_sample = mask_sample.sum()
    if n_full < 10 or n_sample == 0:
        continue
    X_full = df.loc[mask_full, FEATURES_RELATIVAS].values
    idx_all = np.flatnonzero(mask_sample)
    mask_rob = (df['IS_ROBUST_ANOMALY'].values == 1)
    idx_rob = np.flatnonzero(mask_sample & mask_rob)
    idx_norm = np.setdiff1d(idx_all, idx_rob)
    rem = MAX_SHAP_ROWS_PER_GROUP - len(idx_rob)
    rng_shap = np.random.default_rng(SEMILLA_PRINCIPAL + i)
    if rem < 0:
        idx_pos = rng_shap.choice(idx_rob, size=MAX_SHAP_ROWS_PER_GROUP, replace=False)
    elif len(idx_norm) > rem:
        idx_norm_sel = rng_shap.choice(idx_norm, size=rem, replace=False)
        idx_pos = np.concatenate([idx_rob, idx_norm_sel])
    else:
        idx_pos = idx_all
    n_sample = len(idx_pos)
    X_sample = df.iloc[idx_pos][FEATURES_RELATIVAS].values
    model = IsolationForest(**IF_PARAMS)
    model.fit(X_full)
    explainer = shap.TreeExplainer(model)
    sv = explainer.shap_values(X_sample, check_additivity=False)
    sample_idx = df.index[idx_pos]
    for k, idx in enumerate(sample_idx):
        shap_seg_records.append({
            'idx': idx,
            'segmento': seg,
            'shap_seg': sv[k].astype(np.float32),
        })
    if (i + 1) % 10 == 0 or i == len(segmentos) - 1:
        elapsed = time.time() - t0
        print(f'  Seg {seg}: n={n_full:,}, sample={n_sample:,} | {i+1}/{len(segmentos)} ({elapsed:.0f}s)')

print(f'\nNivel 1 completado en {time.time()-t0:.0f}s')
print(f'Filas con SHAP de segmento: {len(shap_seg_records):,}')

Segmentos con observaciones en población SHAP: 57
  Seg 23: n=25,078, sample=1,405 | 10/57 (31s)
  Seg 41: n=123,457, sample=4,000 | 20/57 (91s)
  Seg 51: n=35,887, sample=1,631 | 30/57 (142s)
  Seg 72: n=202,651, sample=4,000 | 40/57 (182s)
  Seg 85: n=32,315, sample=1,414 | 50/57 (223s)
  Seg 95: n=1,477, sample=95 | 57/57 (241s)

Nivel 1 completado en 241s
Filas con SHAP de segmento: 114,491


## 5. SHAP por clase (Nivel 2)

Solo para clases con `n >= 500` (mismo criterio que exp_02). Las anomalías de clases por debajo del umbral usaron únicamente el modelo de segmento por la regla de fallback.

In [7]:
IF_PARAMS_CLASE = {**IF_PARAMS, 'n_jobs': 1}

clase_counts = df['OFERTA_CLASE'].value_counts()
clases_viables = set(clase_counts[clase_counts >= MIN_OBS_CLASE].index.tolist())

shap_cls_records = []
clases_con_sample = (
    df.loc[df['IS_SAMPLE'] == 1, 'OFERTA_CLASE']
    .value_counts()
    .index.tolist()
)
clases_a_correr = [c for c in clases_con_sample if c in clases_viables]
print(f'Clases viables con observaciones en población SHAP: {len(clases_a_correr)}')

t0 = time.time()
for i, clase in enumerate(sorted(clases_a_correr)):
    mask_full = (df['OFERTA_CLASE'] == clase).values
    mask_sample = mask_full & (df['IS_SAMPLE'].values == 1)
    n_full = mask_full.sum()
    n_sample = mask_sample.sum()
    if n_full < MIN_OBS_CLASE or n_sample == 0:
        continue
    X_full = df.loc[mask_full, FEATURES_ABSOLUTAS].values
    idx_all = np.flatnonzero(mask_sample)
    mask_rob = (df['IS_ROBUST_ANOMALY'].values == 1)
    idx_rob = np.flatnonzero(mask_sample & mask_rob)
    idx_norm = np.setdiff1d(idx_all, idx_rob)
    rem = MAX_SHAP_ROWS_PER_GROUP - len(idx_rob)
    rng_shap = np.random.default_rng(SEMILLA_PRINCIPAL + i + 99999)
    if rem < 0:
        idx_pos = rng_shap.choice(idx_rob, size=MAX_SHAP_ROWS_PER_GROUP, replace=False)
    elif len(idx_norm) > rem:
        idx_norm_sel = rng_shap.choice(idx_norm, size=rem, replace=False)
        idx_pos = np.concatenate([idx_rob, idx_norm_sel])
    else:
        idx_pos = idx_all
    n_sample = len(idx_pos)
    X_sample = df.iloc[idx_pos][FEATURES_ABSOLUTAS].values
    model = IsolationForest(**IF_PARAMS_CLASE)
    model.fit(X_full)
    explainer = shap.TreeExplainer(model)
    sv = explainer.shap_values(X_sample, check_additivity=False)
    sample_idx = df.index[idx_pos]
    for k, idx in enumerate(sample_idx):
        shap_cls_records.append({
            'idx': idx,
            'clase': clase,
            'shap_cls': sv[k].astype(np.float32),
        })
    if (i + 1) % 50 == 0 or i == len(clases_a_correr) - 1:
        elapsed = time.time() - t0
        rate = (i + 1) / max(elapsed, 1)
        eta = (len(clases_a_correr) - i - 1) / max(rate, 1e-6)
        print(f'  {i+1}/{len(clases_a_correr)} ({elapsed:.0f}s, ETA {eta:.0f}s)')

print(f'\nNivel 2 completado en {time.time()-t0:.0f}s')
print(f'Filas con SHAP de clase: {len(shap_cls_records):,}')

Clases viables con observaciones en población SHAP: 656
  50/656 (63s, ETA 761s)
  100/656 (142s, ETA 790s)
  150/656 (208s, ETA 702s)
  200/656 (281s, ETA 642s)
  250/656 (345s, ETA 561s)
  300/656 (409s, ETA 485s)
  350/656 (472s, ETA 413s)
  400/656 (546s, ETA 349s)
  450/656 (612s, ETA 280s)
  500/656 (696s, ETA 217s)
  550/656 (762s, ETA 147s)
  600/656 (836s, ETA 78s)
  650/656 (908s, ETA 8s)
  656/656 (916s, ETA 0s)

Nivel 2 completado en 916s
Filas con SHAP de clase: 152,627


## 6. Consolidación: matrices SHAP por nivel

Se construyen dos matrices densas (`shap_seg_matrix` y `shap_cls_matrix`) indexadas por la posición original en `df`. Las observaciones sin SHAP de clase (fallback) llevan vector cero en la matriz de clase, lo cual implica usar exclusivamente el SHAP de segmento para el análisis consolidado.

In [8]:
sample_idx_all = df.index[df['IS_SAMPLE'] == 1]
n_sample_total = len(sample_idx_all)
idx_to_pos = {idx: pos for pos, idx in enumerate(sample_idx_all)}

shap_seg_matrix = np.zeros((n_sample_total, len(FEATURES_RELATIVAS)), dtype=np.float32)
for r in shap_seg_records:
    pos = idx_to_pos.get(r['idx'])
    if pos is not None:
        shap_seg_matrix[pos] = r['shap_seg']

shap_cls_matrix = np.zeros((n_sample_total, len(FEATURES_ABSOLUTAS)), dtype=np.float32)
has_cls_shap = np.zeros(n_sample_total, dtype=bool)
for r in shap_cls_records:
    pos = idx_to_pos.get(r['idx'])
    if pos is not None:
        shap_cls_matrix[pos] = r['shap_cls']
        has_cls_shap[pos] = True

print(f'Población SHAP final: {n_sample_total:,}')
print(f'  Con SHAP de segmento: {(np.abs(shap_seg_matrix).sum(axis=1) > 0).sum():,}')
print(f'  Con SHAP de clase:    {has_cls_shap.sum():,}')
print(f'  Solo segmento (fallback): {(~has_cls_shap).sum():,}')

Población SHAP final: 158,230
  Con SHAP de segmento: 114,491
  Con SHAP de clase:    152,627
  Solo segmento (fallback): 5,603


## 7. Tipologías de anomalía por driver dominante

Cada feature se clasifica en una de cinco categorías temáticas. Para cada anomalía robusta se suma `|SHAP|` por categoría y se asigna como **driver dominante** la categoría con mayor contribución absoluta. Una anomalía es *mixta* si la categoría dominante aporta menos del 40% de la contribución total.

In [9]:
def categorize_features(feature_list, tp_cols, mp_cols):
    cat = {}
    precio = {'RATIO_OFERTADO_VS_ESTIMADO', 'LOG_RATIO_VS_ESTIMADO',
              'RATIO_OFERTADO_VS_PROM_ITEM', 'RATIO_OFERTADO_VS_MIN_ITEM',
              'LOG_PRECIO_OFERTADO', 'LOG_PRECIO_ESTIMADO',
              'OFERTA_IVA', 'OFERTA_DESCUENTO'}
    competencia = {'N_OFERTAS_ITEM', 'N_OFERENTES_ITEM', 'N_OFERENTES_ELEGIBLES_ITEM',
                   'SINGLE_BID_FLAG', 'RANK_PRECIO_ASC', 'CV_PRECIO_ITEM'}
    temporalidad = {'DIAS_PARA_CIERRE', 'DIAS_VENTANA_PROCESO'}
    procedimiento = set(tp_cols) | set(mp_cols) | {'REGIMEN_LGCP'}
    for f in feature_list:
        if f in precio: cat[f] = 'precio'
        elif f in competencia: cat[f] = 'competencia'
        elif f in temporalidad: cat[f] = 'temporalidad'
        elif f in procedimiento: cat[f] = 'procedimiento'
        else: cat[f] = 'otras'
    return cat

cat_rel = categorize_features(FEATURES_RELATIVAS, tp_cols, mp_cols)
cat_abs = categorize_features(FEATURES_ABSOLUTAS, tp_cols, mp_cols)
categorias = ['precio', 'competencia', 'temporalidad', 'procedimiento', 'otras']

def aggregate_categoria(shap_matrix, feature_list, cat_dict, categorias):
    abs_mat = np.abs(shap_matrix)
    agg = np.zeros((shap_matrix.shape[0], len(categorias)), dtype=np.float32)
    for j, f in enumerate(feature_list):
        c = cat_dict[f]
        ci = categorias.index(c)
        agg[:, ci] += abs_mat[:, j]
    return agg

agg_seg = aggregate_categoria(shap_seg_matrix, FEATURES_RELATIVAS, cat_rel, categorias)
agg_cls = aggregate_categoria(shap_cls_matrix, FEATURES_ABSOLUTAS, cat_abs, categorias)
agg_consolidado = np.where(has_cls_shap[:, None], agg_seg + agg_cls, agg_seg)

total_per_row = agg_consolidado.sum(axis=1, keepdims=True)
total_per_row = np.where(total_per_row == 0, 1, total_per_row)
share = agg_consolidado / total_per_row

dominant_idx = share.argmax(axis=1)
dominant_share = share[np.arange(len(share)), dominant_idx]
tipologia = np.array([categorias[i] for i in dominant_idx])
tipologia[dominant_share < 0.40] = 'mixta'

tipologias_disponibles = ['precio', 'competencia', 'temporalidad', 'procedimiento', 'otras', 'mixta']
is_anom_sample = (df.loc[sample_idx_all, 'IS_ROBUST_ANOMALY'].values == 1)

from collections import Counter
ct_anom = Counter(tipologia[is_anom_sample])
ct_norm = Counter(tipologia[~is_anom_sample])

print('Distribución de tipologías sobre anomalías robustas:')
for t in tipologias_disponibles:
    n = ct_anom.get(t, 0)
    print(f'  {t:<15s} {n:>8,} ({n/max(is_anom_sample.sum(),1)*100:>5.1f}%)')
print('\nDistribución de tipologías sobre muestra normal (referencia):')
for t in tipologias_disponibles:
    n = ct_norm.get(t, 0)
    print(f'  {t:<15s} {n:>8,} ({n/max((~is_anom_sample).sum(),1)*100:>5.1f}%)')

Distribución de tipologías sobre anomalías robustas:
  precio            45,026 ( 41.6%)
  competencia       34,378 ( 31.8%)
  temporalidad      15,950 ( 14.7%)
  procedimiento         44 (  0.0%)
  otras              1,063 (  1.0%)
  mixta             11,770 ( 10.9%)

Distribución de tipologías sobre muestra normal (referencia):
  precio             5,923 ( 11.8%)
  competencia       19,555 ( 39.1%)
  temporalidad       2,595 (  5.2%)
  procedimiento         35 (  0.1%)
  otras                712 (  1.4%)
  mixta             21,179 ( 42.4%)


## 8. Importancia global de features

La importancia global se calcula como el promedio de `|SHAP|` por feature, restringido al conjunto de **anomalías robustas** (no toda la población SHAP), porque interesa entender qué impulsa los casos atípicos, no la población general.

In [10]:
abs_seg_anom = np.abs(shap_seg_matrix[is_anom_sample])
imp_seg = abs_seg_anom.mean(axis=0)
imp_seg_df = pd.DataFrame({'feature': FEATURES_RELATIVAS, 'importancia_seg': imp_seg,
                            'categoria': [cat_rel[f] for f in FEATURES_RELATIVAS]})
imp_seg_df = imp_seg_df.sort_values('importancia_seg', ascending=False)

anom_with_cls = is_anom_sample & has_cls_shap
if anom_with_cls.sum() > 0:
    abs_cls_anom = np.abs(shap_cls_matrix[anom_with_cls])
    imp_cls = abs_cls_anom.mean(axis=0)
    imp_cls_df = pd.DataFrame({'feature': FEATURES_ABSOLUTAS, 'importancia_cls': imp_cls,
                                'categoria': [cat_abs[f] for f in FEATURES_ABSOLUTAS]})
    imp_cls_df = imp_cls_df.sort_values('importancia_cls', ascending=False)
else:
    imp_cls_df = pd.DataFrame(columns=['feature', 'importancia_cls', 'categoria'])

print('Top 15 features por importancia SHAP (Nivel 1 - Segmento):')
print(imp_seg_df.head(15).to_string(index=False))
print('\nTop 15 features por importancia SHAP (Nivel 2 - Clase):')
print(imp_cls_df.head(15).to_string(index=False))

Top 15 features por importancia SHAP (Nivel 1 - Segmento):
                    feature  importancia_seg     categoria
 RATIO_OFERTADO_VS_ESTIMADO         0.582094        precio
       DIAS_VENTANA_PROCESO         0.519514  temporalidad
      LOG_RATIO_VS_ESTIMADO         0.518544        precio
            SINGLE_BID_FLAG         0.355218   competencia
           DIAS_PARA_CIERRE         0.321570  temporalidad
             N_OFERTAS_ITEM         0.289688   competencia
                 OFERTA_IVA         0.285305        precio
             CV_PRECIO_ITEM         0.278501   competencia
           N_OFERENTES_ITEM         0.250921   competencia
            RANK_PRECIO_ASC         0.232820   competencia
RATIO_OFERTADO_VS_PROM_ITEM         0.231907        precio
 N_OFERENTES_ELEGIBLES_ITEM         0.224768   competencia
 RATIO_OFERTADO_VS_MIN_ITEM         0.219575        precio
       TAMANO_PROVEEDOR_ORD         0.141477         otras
               REGIMEN_LGCP         0.119216 procedimien

## 9. Figuras de comunicación

In [11]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

top_seg = imp_seg_df.head(15)
colors_map = {'precio': '#d62728', 'competencia': '#2ca02c',
              'temporalidad': '#ff7f0e', 'procedimiento': '#9467bd', 'otras': '#7f7f7f'}
axes[0].barh(top_seg['feature'][::-1], top_seg['importancia_seg'][::-1],
             color=[colors_map[c] for c in top_seg['categoria'][::-1]])
axes[0].set_title('Importancia SHAP — Nivel 1 (Segmento)', fontsize=13)
axes[0].set_xlabel('|SHAP| promedio sobre anomalías robustas')

if not imp_cls_df.empty:
    top_cls = imp_cls_df.head(15)
    axes[1].barh(top_cls['feature'][::-1], top_cls['importancia_cls'][::-1],
                 color=[colors_map[c] for c in top_cls['categoria'][::-1]])
    axes[1].set_title('Importancia SHAP — Nivel 2 (Clase)', fontsize=13)
    axes[1].set_xlabel('|SHAP| promedio sobre anomalías robustas con clase')

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=v, label=k.capitalize()) for k, v in colors_map.items()]
fig.legend(handles=legend_elements, loc='lower center', ncol=5, bbox_to_anchor=(0.5, -0.02))
fig.suptitle('Drivers de las anomalías — Importancia global de features (SHAP)', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'shap_importancia_global.png', dpi=120, bbox_inches='tight')
plt.close()
print(f'Guardado: {FIG_DIR / "shap_importancia_global.png"}')

Guardado: figuras\shap_importancia_global.png


In [12]:
feature_values_seg = df.loc[sample_idx_all, FEATURES_RELATIVAS].values[is_anom_sample]
shap_values_seg_anom = shap_seg_matrix[is_anom_sample]
top_feat_idx = np.argsort(-np.abs(shap_values_seg_anom).mean(axis=0))[:15]
top_feat_names = [FEATURES_RELATIVAS[i] for i in top_feat_idx]

plt.figure(figsize=(12, 8))
shap.summary_plot(
    shap_values_seg_anom[:, top_feat_idx],
    feature_values_seg[:, top_feat_idx],
    feature_names=top_feat_names,
    show=False, plot_size=None,
)
plt.title('Beeswarm SHAP — Anomalías robustas (Nivel 1: Segmento)', fontsize=13, pad=15)
plt.tight_layout()
plt.savefig(FIG_DIR / 'shap_beeswarm.png', dpi=120, bbox_inches='tight')
plt.close()
print(f'Guardado: {FIG_DIR / "shap_beeswarm.png"}')

Guardado: figuras\shap_beeswarm.png


In [13]:
anom_tipo_counts = pd.Series(tipologia[is_anom_sample]).value_counts().reindex(tipologias_disponibles, fill_value=0)
norm_tipo_counts = pd.Series(tipologia[~is_anom_sample]).value_counts().reindex(tipologias_disponibles, fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
color_tipo = {'precio': '#d62728', 'competencia': '#2ca02c', 'temporalidad': '#ff7f0e',
              'procedimiento': '#9467bd', 'otras': '#7f7f7f', 'mixta': '#1f77b4'}

anom_pct = (anom_tipo_counts / max(anom_tipo_counts.sum(), 1) * 100)
axes[0].bar(anom_pct.index, anom_pct.values, color=[color_tipo[t] for t in anom_pct.index])
axes[0].set_title(f'Tipologías sobre anomalías robustas (n={anom_tipo_counts.sum():,})', fontsize=12)
axes[0].set_ylabel('% del total')
for i, (t, v) in enumerate(anom_pct.items()):
    axes[0].text(i, v + 0.3, f'{v:.1f}%', ha='center', fontsize=10)
axes[0].tick_params(axis='x', rotation=20)

norm_pct = (norm_tipo_counts / max(norm_tipo_counts.sum(), 1) * 100)
axes[1].bar(norm_pct.index, norm_pct.values, color=[color_tipo[t] for t in norm_pct.index])
axes[1].set_title(f'Tipologías sobre muestra normal (n={norm_tipo_counts.sum():,})', fontsize=12)
axes[1].set_ylabel('% del total')
for i, (t, v) in enumerate(norm_pct.items()):
    axes[1].text(i, v + 0.3, f'{v:.1f}%', ha='center', fontsize=10)
axes[1].tick_params(axis='x', rotation=20)

fig.suptitle('Distribución de tipologías por driver dominante (SHAP, Nivel 1)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'tipologias_anomalia.png', dpi=120, bbox_inches='tight')
plt.close()
print(f'Guardado: {FIG_DIR / "tipologias_anomalia.png"}')

Guardado: figuras\tipologias_anomalia.png


## 10. Waterfalls representativos

Para cada tipología (precio, competencia, temporalidad, procedimiento, mixta) se selecciona la anomalía más extrema (mayor `|SHAP|` total) y se grafica su waterfall.

In [14]:
abs_total_anom = np.abs(shap_seg_matrix).sum(axis=1)

for t in ['precio', 'competencia', 'temporalidad', 'procedimiento', 'mixta']:
    mask_t = is_anom_sample & (tipologia == t)
    if mask_t.sum() == 0:
        print(f'  Tipología "{t}": sin observaciones, se omite waterfall.')
        continue
    cand_pos = np.where(mask_t)[0]
    chosen_pos = cand_pos[np.argmax(abs_total_anom[cand_pos])]
    sv_row = shap_seg_matrix[chosen_pos]
    fv_row = df.loc[sample_idx_all[chosen_pos], FEATURES_RELATIVAS].values

    top_idx = np.argsort(-np.abs(sv_row))[:12]
    sv_top = sv_row[top_idx]
    fv_top = fv_row[top_idx]
    names_top = [FEATURES_RELATIVAS[i] for i in top_idx]
    label_top = [f'{n} = {v:.2f}' for n, v in zip(names_top, fv_top)]

    fig, ax = plt.subplots(figsize=(11, 7))
    colors = ['#d62728' if v < 0 else '#2ca02c' for v in sv_top]
    ax.barh(label_top[::-1], sv_top[::-1], color=colors[::-1])
    ax.axvline(0, color='black', lw=0.8)
    ax.set_xlabel('Valor SHAP (negativo → empuja hacia anomalía)')
    ax.set_title(f'Anomalía representativa — tipología "{t}"', fontsize=12)
    plt.tight_layout()
    out = FIG_DIR / f'shap_waterfall_{t}.png'
    plt.savefig(out, dpi=120, bbox_inches='tight')
    plt.close()
    print(f'  Guardado: {out}')

  Guardado: figuras\shap_waterfall_precio.png
  Guardado: figuras\shap_waterfall_competencia.png
  Guardado: figuras\shap_waterfall_temporalidad.png
  Guardado: figuras\shap_waterfall_procedimiento.png
  Guardado: figuras\shap_waterfall_mixta.png


## 11. Exportar resultados

In [15]:
out_drivers = pd.DataFrame({
    'idx_orig': sample_idx_all,
    'is_robust_anomaly': is_anom_sample.astype(int),
    'tipologia_dominante': tipologia,
    'share_dominante': share[np.arange(len(share)), dominant_idx],
    'shap_abs_total_seg': np.abs(shap_seg_matrix).sum(axis=1),
    'shap_abs_total_cls': np.abs(shap_cls_matrix).sum(axis=1),
    'has_cls_shap': has_cls_shap.astype(int),
})
for ci, c in enumerate(categorias):
    out_drivers[f'share_{c}'] = share[:, ci]

key_subset = df.loc[sample_idx_all, key_cols + ['OFERTA_SEGMENTO', 'OFERTA_CLASE',
                                                  'NOMBRE_INSTITUCION', 'NOMBRE_PROVEEDOR',
                                                  'OFERTA_PRECIO_UNITARIO_CRC',
                                                  'RATIO_OFERTADO_VS_ESTIMADO',
                                                  'N_OFERENTES_ITEM', 'SINGLE_BID_FLAG']].reset_index(drop=True)
out_drivers = out_drivers.reset_index(drop=True)
out_drivers = pd.concat([key_subset, out_drivers.drop(columns=['idx_orig'])], axis=1)

out_path = RES_DIR / 'anomalias_con_drivers.parquet'
out_drivers.to_parquet(out_path, index=False)
print(f'Guardado: {out_path}  ({len(out_drivers):,} filas)')

shap_df_seg = pd.DataFrame(shap_seg_matrix, columns=[f'shap_seg__{f}' for f in FEATURES_RELATIVAS])
shap_df_cls = pd.DataFrame(shap_cls_matrix, columns=[f'shap_cls__{f}' for f in FEATURES_ABSOLUTAS])
shap_export = pd.concat([key_subset[key_cols].reset_index(drop=True), shap_df_seg, shap_df_cls], axis=1)
shap_export.to_parquet(RES_DIR / 'shap_values_top.parquet', index=False)
print(f'Guardado: {RES_DIR / "shap_values_top.parquet"}  ({len(shap_export):,} filas, {shap_export.shape[1]} columnas)')

imp_seg_df.to_csv(RES_DIR / 'shap_importancia_segmento.csv', index=False)
imp_cls_df.to_csv(RES_DIR / 'shap_importancia_clase.csv', index=False)
print(f'Importancias guardadas en {RES_DIR}/')

Guardado: resultados\anomalias_con_drivers.parquet  (158,230 filas)
Guardado: resultados\shap_values_top.parquet  (158,230 filas, 39 columnas)
Importancias guardadas en resultados/


## 12. Resumen ejecutivo del experimento

In [ ]:
print('=' * 70)
print('RESUMEN EJECUTIVO — Experimento 03 (SHAP)')
print('=' * 70)
print(f'Anomalías robustas analizadas:    {is_anom_sample.sum():,}')
print(f'Muestra normal de referencia:     {(~is_anom_sample).sum():,}')
print(f'Modelos de segmento explicados:   {len(set(r["segmento"] for r in shap_seg_records))}')
print(f'Modelos de clase explicados:      {len(set(r["clase"] for r in shap_cls_records))}')
print()
print('Tipologías de anomalía (driver dominante):')
for t in tipologias_disponibles:
    n = ct_anom.get(t, 0)
    pct = n / max(is_anom_sample.sum(), 1) * 100
    print(f'  {t:<15s} {n:>8,} ({pct:>5.1f}%)')
print()
print('Top 5 features por importancia SHAP (Nivel 1):')
for _, r in imp_seg_df.head(5).iterrows():
    print(f'  {r["feature"]:<35s} {r["importancia_seg"]:.4f}  ({r["categoria"]})')
print()
print('Artefactos:')
for p in sorted(FIG_DIR.glob('shap_*')) + sorted(FIG_DIR.glob('tipologias_*')):
    print(f'  {p}')
for p in sorted(RES_DIR.glob('*')):
    print(f'  {p}')

RESUMEN EJECUTIVO — Experimento 03 (SHAP)
Anomalías robustas analizadas:    108,231
Muestra normal de referencia:     49,999
Modelos de segmento explicados:   57
Modelos de clase explicados:      656

Tipologías de anomalía (driver dominante):
  precio            45,026 ( 41.6%)
  competencia       34,378 ( 31.8%)
  temporalidad      15,950 ( 14.7%)
  procedimiento         44 (  0.0%)
  otras              1,063 (  1.0%)
  mixta             11,770 ( 10.9%)

Top 5 features por importancia SHAP (Nivel 1):
  RATIO_OFERTADO_VS_ESTIMADO          0.5821  (precio)
  DIAS_VENTANA_PROCESO                0.5195  (temporalidad)
  LOG_RATIO_VS_ESTIMADO               0.5185  (precio)
  SINGLE_BID_FLAG                     0.3552  (competencia)
  DIAS_PARA_CIERRE                    0.3216  (temporalidad)

Artefactos:
  figuras\shap_beeswarm.png
  figuras\shap_importancia_global.png
  figuras\shap_waterfall_competencia.png
  figuras\shap_waterfall_mixta.png
  figuras\shap_waterfall_precio.png
  figuras

: 